# 图像几何变换实验练习版：仿射变换、透视变换、旋转、镜像和平移

本 notebook 是学生练习版本。你需要补全关键函数，实现图像的仿射变换、透视变换、旋转、镜像和平移，为后续学习图像配准和张正友标定法做准备。

要求：

- 使用齐次坐标表示二维几何变换。
- 使用目标像素到源图像坐标的查找方法完成图像变换。
- 不调用第三方库中的 `warp`、`rotate` 或透视变换函数。
- 补全平移、旋转、镜像、仿射变换和透视变换相关函数。
- 实现旋转时自动扩展画布，避免图像内容被裁剪。

## 相关知识

### 1. 齐次坐标

二维图像点 $(x, y)$ 可以写成齐次坐标：

$$\mathbf{p}=\begin{bmatrix}x \\ y \\ 1\end{bmatrix}$$

经过 $3 \times 3$ 变换矩阵 $H$ 后：

$$\begin{bmatrix}x' \\ y' \\ w'\end{bmatrix}=H\begin{bmatrix}x \\ y \\ 1\end{bmatrix}$$

归一化后得到图像坐标：

$$x_{img}=\frac{x'}{w'}, \quad y_{img}=\frac{y'}{w'}$$

### 2. 仿射变换

仿射变换可以表示平移、旋转、缩放、错切等操作。它保持直线仍为直线，保持平行线仍然平行。矩阵形式为：

$$A=\begin{bmatrix}a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \\ 0 & 0 & 1\end{bmatrix}$$

对应坐标关系为：

$$x'=a_{11}x+a_{12}y+t_x$$

$$y'=a_{21}x+a_{22}y+t_y$$

### 3. 透视变换

透视变换也称单应性变换 Homography，可以描述平面在不同视角下的投影关系：

$$H=\begin{bmatrix}h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33}\end{bmatrix}$$

$$x'=\frac{h_{11}x+h_{12}y+h_{13}}{h_{31}x+h_{32}y+h_{33}}$$

$$y'=\frac{h_{21}x+h_{22}y+h_{23}}{h_{31}x+h_{32}y+h_{33}}$$

张正友标定法会利用平面棋盘格角点在世界平面和图像平面之间的对应关系估计单应矩阵。

### 4. 仿射变换和透视变换的不同之处

- 仿射变换有 6 个自由度，透视变换有 8 个自由度。
- 仿射变换保持平行线仍然平行；透视变换不一定保持平行线平行。
- 仿射变换可以把矩形变成平行四边形；透视变换可以把矩形变成任意凸四边形。
- 仿射变换适合视角变化较小的近似对齐；透视变换适合描述平面物体在不同视角下的成像关系。
- 张正友标定法中的棋盘格平面到图像平面的关系通常是透视变换，而不是简单仿射变换。

### 5. 目标像素到源图像坐标的查找

遍历目标图像中的每个像素，利用变换矩阵的逆矩阵找到它在原图像中的对应位置：

$$\begin{bmatrix}x \\ y \\ w\end{bmatrix}=H^{-1}\begin{bmatrix}x' \\ y' \\ 1\end{bmatrix}$$

$$x_s=\frac{x}{w}, \quad y_s=\frac{y}{w}$$

源坐标为浮点数时，本实验用四舍五入找到最近的原图像素：

$$x_n=\text{round}(x_s), \quad y_n=\text{round}(y_s)$$

如果最近像素坐标超出原图范围，则填充背景值。

## 公共代码

请先补全 `nearest_sample` 和 `warp_image`，后续所有几何变换都会依赖它们。

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, img_as_float, io
from skimage.color import rgba2rgb


def load_image():
    """优先读取本地图像；如果没有，则使用内置图像作为备用。"""
    candidate_paths = [
        Path("lena.png"), Path("lena.jpg"), Path("lena.jpeg"), Path("lena.bmp"),
        Path("Lenna.png"), Path("Lenna.jpg"), Path("Lenna.jpeg"), Path("Lenna.bmp"),
        Path("image.png"), Path("image.jpg"), Path("input.png"), Path("input.jpg"),
    ]

    for path in candidate_paths:
        if path.exists():
            image = img_as_float(io.imread(path))
            break
    else:
        image = img_as_float(data.astronaut())

    if image.ndim == 3 and image.shape[-1] == 4:
        image = rgba2rgb(image)
    return image


def ensure_3d(image):
    if image.ndim == 2:
        return image[..., np.newaxis], True
    return image, False


def restore_shape(image, was_gray):
    if was_gray:
        return image[..., 0]
    return image


def nearest_sample(image, x, y, background=0.0):
    """根据浮点源坐标取最近的原图像素。

    学生需要补全：
    1. 读取图像高度、宽度和通道数。
    2. 对浮点坐标 x、y 四舍五入，得到 nearest_x、nearest_y。
    3. 判断最近像素是否超出图像范围。
    4. 如果越界，返回背景值组成的通道向量。
    5. 如果合法，返回 image[nearest_y, nearest_x]。
    """
    # TODO：请补全最近像素取值函数。
    raise NotImplementedError("请补全 nearest_sample。")


def warp_image(image, transform, output_shape=None, background=0.0):
    """使用目标到源坐标查找完成图像变换。

    transform 表示源图像坐标到目标图像坐标的矩阵。

    学生需要补全：
    1. 使用 ensure_3d 统一处理灰度图和彩色图。
    2. 如果 output_shape 为 None，则输出尺寸等于原图尺寸。
    3. 计算变换矩阵的逆矩阵 inverse_transform。
    4. 遍历目标图像每一个像素 target_x、target_y。
    5. 构造目标齐次坐标 [target_x, target_y, 1]。
    6. 用 inverse_transform 得到源图像齐次坐标。
    7. 对齐次坐标做归一化，得到 source_x、source_y。
    8. 调用 nearest_sample 得到像素值。
    9. 使用 restore_shape 恢复灰度图形状并返回结果。
    """
    # TODO：请补全图像几何变换主函数。
    raise NotImplementedError("请补全 warp_image。")


def show_images(images, titles, cols=3):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 4.2 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(np.clip(img, 0, 1), cmap="gray" if img.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


image = load_image()
show_images([image], ["Original Image"], cols=1)

## 代码段 1：平移变换

请补全 `translation_matrix`。

大致步骤：

1. 构造 $3 \times 3$ 单位矩阵。
2. 将第一行第三列设置为 `tx`。
3. 将第二行第三列设置为 `ty`。
4. 返回平移矩阵。

In [ ]:
def translation_matrix(tx, ty):
    """构造平移矩阵。"""
    # TODO：请补全平移矩阵。
    raise NotImplementedError("请补全 translation_matrix。")


tx = 70
ty = 35
translated = warp_image(image, translation_matrix(tx, ty), background=0.0)

show_images([image, translated], ["Original Image", "Translation"], cols=2)

## 代码段 2：镜像变换

请补全 `mirror_matrix`。

大致步骤：

1. 计算图像中心点 `center_x`、`center_y`。
2. 构造将中心移到原点的平移矩阵。
3. 构造水平镜像或垂直镜像矩阵。
4. 构造将中心移回原位置的平移矩阵。
5. 按“移回中心 @ 镜像 @ 移到原点”的顺序组合矩阵。

In [ ]:
def mirror_matrix(width, height, mode="horizontal"):
    """构造绕图像中心的水平或垂直镜像矩阵。"""
    # TODO：请补全镜像矩阵。
    raise NotImplementedError("请补全 mirror_matrix。")


height, width = image.shape[:2]
horizontal_mirror = warp_image(image, mirror_matrix(width, height, "horizontal"))
vertical_mirror = warp_image(image, mirror_matrix(width, height, "vertical"))

show_images(
    [image, horizontal_mirror, vertical_mirror],
    ["Original Image", "Horizontal Mirror", "Vertical Mirror"],
    cols=3
)

## 代码段 3：旋转变换与扩展画布

请补全 `rotation_matrix`、`transform_points` 和 `expanded_rotation_transform`。

大致步骤：

1. `rotation_matrix`：将角度转为弧度，构造绕原点旋转矩阵，再结合平移矩阵实现绕图像中心旋转。
2. `transform_points`：把二维点扩展为齐次坐标，左乘变换矩阵，再除以第三个分量得到二维坐标。
3. `expanded_rotation_transform`：计算原图四个角点旋转后的位置。
4. 根据旋转后角点的最小/最大坐标确定新画布大小。
5. 增加平移矩阵，把旋转后的图像移动到新画布的正坐标区域。
6. 返回扩展画布下的变换矩阵和输出尺寸。

In [ ]:
def rotation_matrix(angle_degrees, center_x, center_y):
    """构造绕指定中心旋转的矩阵。"""
    # TODO：请补全旋转矩阵。
    raise NotImplementedError("请补全 rotation_matrix。")


def transform_points(points, transform):
    """对二维点集应用齐次变换。"""
    # TODO：请补全点集变换。
    raise NotImplementedError("请补全 transform_points。")


def expanded_rotation_transform(width, height, angle_degrees):
    """构造扩展画布后的旋转矩阵和输出尺寸。"""
    # TODO：请补全扩展画布旋转逻辑。
    raise NotImplementedError("请补全 expanded_rotation_transform。")


angle = 30
height, width = image.shape[:2]
rotated_cropped = warp_image(image, rotation_matrix(angle, (width - 1) / 2, (height - 1) / 2), background=0.0)
expanded_transform, expanded_shape = expanded_rotation_transform(width, height, angle)
rotated_expanded = warp_image(image, expanded_transform, output_shape=expanded_shape, background=0.0)

show_images(
    [image, rotated_cropped, rotated_expanded],
    ["Original Image", "Rotation Same Canvas", "Rotation Expanded Canvas"],
    cols=3
)
print("Expanded output shape:", expanded_shape)

## 代码段 4：一般仿射变换

请补全 `affine_matrix`。

大致步骤：

1. 将 `scale_x`、`scale_y` 放在主对角线位置。
2. 将 `shear_x` 放在第一行第二列。
3. 将 `shear_y` 放在第二行第一列。
4. 将 `tx`、`ty` 放在平移位置。
5. 最后一行应为 `[0, 0, 1]`。

In [ ]:
def affine_matrix(scale_x=1.0, scale_y=1.0, shear_x=0.0, shear_y=0.0, tx=0.0, ty=0.0):
    """构造一般仿射变换矩阵。"""
    # TODO：请补全仿射矩阵。
    raise NotImplementedError("请补全 affine_matrix。")


affine = affine_matrix(scale_x=0.92, scale_y=1.08, shear_x=0.18, shear_y=0.04, tx=-20, ty=15)
affine_result = warp_image(image, affine, background=0.0)

show_images([image, affine_result], ["Original Image", "Affine Transform"], cols=2)

## 代码段 5：透视变换

请补全 `perspective_matrix_from_four_points`。

大致步骤：

1. 输入 4 个源点和 4 个目标点。
2. 假设 $h_{33}=1$，未知量为 8 个。
3. 每组点对应可以列出两条线性方程。
4. 4 组点一共得到 8 条方程。
5. 使用 `np.linalg.solve` 求解 8 个未知量。
6. 将解组装为 $3 \times 3$ 单应矩阵。

In [ ]:
def perspective_matrix_from_four_points(src_points, dst_points):
    """根据四组点对应估计单应矩阵 H，使得 dst ~ H @ src。"""
    # TODO：请补全四点估计透视变换矩阵。
    raise NotImplementedError("请补全 perspective_matrix_from_four_points。")


height, width = image.shape[:2]
source_corners = np.array([
    [0, 0],
    [width - 1, 0],
    [width - 1, height - 1],
    [0, height - 1]
], dtype=float)
target_corners = np.array([
    [70, 30],
    [width - 90, 5],
    [width - 35, height - 45],
    [35, height - 10]
], dtype=float)

homography = perspective_matrix_from_four_points(source_corners, target_corners)
perspective_result = warp_image(image, homography, background=0.0)

show_images([image, perspective_result], ["Original Image", "Perspective Transform"], cols=2)
print("Homography matrix:")
print(homography)

## 代码段 6：图像配准与张正友标定法准备

完成前面的单应矩阵估计函数后，运行下面代码，观察平面点到图像点的对应关系。

In [ ]:
def draw_point_correspondence(src_points, dst_points):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].set_title("Plane Points")
    axes[1].set_title("Image Points")

    for ax, pts in zip(axes, [src_points, dst_points]):
        closed = np.vstack([pts, pts[0]])
        ax.plot(closed[:, 0], closed[:, 1], "o-", linewidth=2)
        for idx, (x, y) in enumerate(pts):
            ax.text(x + 3, y + 3, str(idx), fontsize=12)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.grid(True)

    plt.tight_layout()
    plt.show()


plane_points = np.array([[0, 0], [160, 0], [160, 110], [0, 110]], dtype=float)
image_points = np.array([[35, 25], [190, 12], [172, 135], [18, 115]], dtype=float)
H_plane_to_image = perspective_matrix_from_four_points(plane_points, image_points)

draw_point_correspondence(plane_points, image_points)
print("Plane-to-image homography:")
print(H_plane_to_image)

test_point = np.array([80, 55, 1.0])
mapped = H_plane_to_image @ test_point
mapped = mapped[:2] / mapped[2]
print("Plane center maps to image point:", mapped)

## 检查建议

完成代码后，可以按下面方式检查：

- 平移后图像应整体向右、向下移动。
- 水平镜像应左右翻转，垂直镜像应上下翻转。
- 原尺寸旋转结果可能被裁剪，扩展画布旋转结果应保留完整内容。
- 仿射变换应出现缩放、错切和平移组合效果。
- 透视变换应将矩形图像区域变成类似梯形的投影效果。
- 四点单应矩阵估计应能把平面四边形映射到图像四边形。